# Model comparison: cross-validated R² across the main models

Compares the main model labels (see `modelz` below) on cross-validated R² (mean and median across voxels with cvR² > 0) in NPCr, using the long-format summary table `derivatives/summary_tsvs/main_models_roi-NPCr_desc-groundtruth_parameters.tsv.gz` (written by `write_parameters_summary.py`), split by ground-truth vs. response fits.

In [2]:
from pathlib import Path
import pandas as pd
from neural_priors.utils.data import Subject, get_all_subject_ids
from tqdm.contrib.itertools import product

In [ ]:
modelz = {
    0:  'No shift (μ_wide = μ_narrow)',
    1:  'Fixed shift (δ=2, no range constraint)',
    2:  'Fitted shift ratio, shared across voxels',
    3:  'Fitted shift ratio, free per voxel',
    4:  'Efficient coding: fixed shift (δ=2)',
    5:  'Efficient coding: shared shift ratio',
    14: 'Free width ratio, per voxel',
    15: 'Fitted width scaling, shared across voxels',
    31: 'Fixed width scaling (δ_σ=1.29)',
    32: 'Fixed width scaling + free amplitude ratio',
    33: 'Fixed width scaling + shared amplitude ratio',
    34: 'Fixed tuning, free amplitude per voxel',
    35: 'Fixed tuning, shared amplitude ratio',
}

In [ ]:
bids_folder = Path('/data/ds-neuralpriors')

pars = pd.read_csv(
    bids_folder / 'derivatives' / 'summary_tsvs' / 'main_models_roi-NPCr_desc-groundtruth_parameters.tsv.gz',
    sep='\t')

cvr2 = pars[['subject', 'model_label', 'model', 'response_fit', 'voxel', 'cvr2']]
cvr2 = cvr2[cvr2['cvr2'] > 0]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

order = [modelz[k] for k in sorted(modelz) if modelz[k] in cvr2['model'].unique()]

for response_fit, label in [(False, 'ground truth'), (True, 'responses')]:
    g = sns.catplot(
        data=cvr2[cvr2['response_fit'] == response_fit].groupby(['subject', 'model'])['cvr2'].mean().reset_index(),
        x='model', y='cvr2', kind='point', errorbar='se',
        order=order, aspect=2.0, height=7)
    g.set(title=f'Mean cvr2 — {label}')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
for response_fit, label in [(False, 'ground truth'), (True, 'responses')]:
    g = sns.catplot(
        data=cvr2[cvr2['response_fit'] == response_fit].groupby(['subject', 'model'])['cvr2'].median().reset_index(),
        x='model', y='cvr2', kind='point', errorbar='se',
        order=order, aspect=2.0, height=7)
    g.set(title=f'Median cvr2 — {label}')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()